# Bronze Ingest

This notebook reads the uploaded Instacart-shaped CSV files with explicit schemas and persists them as bronze Delta tables.

## Run Inputs
- `catalog`: optional override for the active catalog
- `schema`: schema that will receive bronze tables
- `raw_volume`: volume that contains the sampled CSV uploads

## Source Inputs
- `orders.csv`
- `order_products__prior.csv`
- `order_products__train.csv`
- `products.csv`
- `aisles.csv`
- `departments.csv`

## Output Tables
- `bronze_orders`
- `bronze_order_products_prior`
- `bronze_order_products_train`
- `bronze_products`
- `bronze_aisles`
- `bronze_departments`

## Workflow
1. Resolve the raw volume path.
2. Define explicit schemas for every uploaded CSV.
3. Read each file with its fixed schema.
4. Overwrite the bronze Delta tables and validate row counts.



## Capture Runtime Inputs

Explicit schemas keep bronze typing stable and avoid CSV inference drift across reruns or environments.



In [ ]:
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType

dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "retailpulse")
dbutils.widgets.text("raw_volume", "retailpulse_raw")



## Resolve Catalog Context

The helper function keeps fully qualified Delta table names consistent with later notebooks.



In [ ]:
catalog = dbutils.widgets.get("catalog") or spark.sql("SELECT current_catalog()").first()[0]
schema = dbutils.widgets.get("schema") or spark.sql("SELECT current_schema()").first()[0]
raw_volume = dbutils.widgets.get("raw_volume")
raw_root = f"/Volumes/{catalog}/{schema}/{raw_volume}"


def qname(name: str) -> str:
    return f"`{catalog}`.`{schema}`.`{name}`"


def read_csv(file_name: str, schema_def: StructType):
    return (
        spark.read.option("header", True)
        .schema(schema_def)
        .csv(f"{raw_root}/{file_name}")
    )


orders_schema = StructType(
    [
        StructField("order_id", IntegerType(), False),
        StructField("user_id", IntegerType(), False),
        StructField("eval_set", StringType(), False),
        StructField("order_number", IntegerType(), False),
        StructField("order_dow", IntegerType(), True),
        StructField("order_hour_of_day", IntegerType(), True),
        StructField("days_since_prior_order", DoubleType(), True),
    ]
)
order_products_schema = StructType(
    [
        StructField("order_id", IntegerType(), False),
        StructField("product_id", IntegerType(), False),
        StructField("add_to_cart_order", IntegerType(), True),
        StructField("reordered", IntegerType(), True),
    ]
)
products_schema = StructType(
    [
        StructField("product_id", IntegerType(), False),
        StructField("product_name", StringType(), True),
        StructField("aisle_id", IntegerType(), True),
        StructField("department_id", IntegerType(), True),
    ]
)
aisles_schema = StructType(
    [
        StructField("aisle_id", IntegerType(), False),
        StructField("aisle", StringType(), True),
    ]
)
departments_schema = StructType(
    [
        StructField("department_id", IntegerType(), False),
        StructField("department", StringType(), True),
    ]
)



## Load Bronze DataFrames

Each source file is read once with an explicit schema and mapped directly to its bronze table target.



In [ ]:
bronze_frames = {
    "bronze_orders": read_csv("orders.csv", orders_schema),
    "bronze_order_products_prior": read_csv("order_products__prior.csv", order_products_schema),
    "bronze_order_products_train": read_csv("order_products__train.csv", order_products_schema),
    "bronze_products": read_csv("products.csv", products_schema),
    "bronze_aisles": read_csv("aisles.csv", aisles_schema),
    "bronze_departments": read_csv("departments.csv", departments_schema),
}



## Persist Bronze Tables

The row-count assertion blocks accidental overwrites with empty inputs, which is the most common failure mode after a bad upload.



In [ ]:
for table_name, frame in bronze_frames.items():
    assert frame.count() > 0, f"{table_name} is empty."
    (
        frame.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(qname(table_name))
    )



## Review Row Counts

Use this output to confirm that the bronze layer is populated before moving on to type standardization in `03_silver_transform`.



In [ ]:
counts = [(name, spark.table(qname(name)).count()) for name in bronze_frames]
display(spark.createDataFrame(counts, ["table_name", "row_count"]))
